# EmbAlign Single Frame Inference Walkthrough

In [1]:
import os
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aligner.config import PipelineConfig
from aligner.matcher import SinkhornMatcher, HungarianMatcher
from aligner.transformer import RigidTransformer
from aligner.engine import ModularAlignmentEngine
from aligner.models import EmbryoFrame
from aligner.oracle import DiagnosticLayer
from aligner.plot_utils import * 
from aligner.runner import *
from aligner.report_builder import *

%load_ext autoreload
%autoreload 2

In [7]:
MODEL_DIR = "production_models"

master_atlas = joblib.load(os.path.join(MODEL_DIR, 'master_gp_atlas.pkl'))
master_slice_db = joblib.load(os.path.join(MODEL_DIR, 'master_slice_db.pkl'))
life_history_df = joblib.load(os.path.join(MODEL_DIR, 'master_life_history.pkl'))
oracle = DiagnosticLayer(model_path=os.path.join(MODEL_DIR, 'production_oracle.pkl'))

DiagnosticLayer: Loaded model from production_models/production_oracle.pkl


In [8]:
config = PipelineConfig.v3_0_production()
engine = ModularAlignmentEngine(
    config=config,
    atlas=master_atlas,
    slice_db=master_slice_db,
    coarse_matcher=HungarianMatcher(tau=config.tau),
    icp_matcher=SinkhornMatcher(
        epsilon=config.epsilon_refine, 
        stop_thr=config.sinkhorn_stop_thr,
        max_iters=config.sinkhorn_max_iters
    ),
    transformer=RigidTransformer()
)

## Read in nuclei centroid positions
- Input is csv containg X, Y, and Z coordines.


In [9]:
TARGET_CSV = "/Users/miles/Documents/UCLA/shah_lab/embryo_aligner/notebooks/refactor_testing/inference_data_v2/Pos106_ShortWavelengthCamera_reg1_emb1.csv"

nuclei_csv = pd.read_csv(TARGET_CSV)
nuclei_csv.head()


,X,Y,Z,D
0,93.21,108.84,33.914,65.625
1,121.88,274.39,34.172,65.625
2,241.03,341.38,33.508,65.625
3,237.14,187.58,35.004,65.625
4,214.66,110.21,38.380,65.625


In [10]:
# Initialize embryo frame from nuclei positions. 
# Scope specific scale factors can also be input here.
SCALE_XY = 0.1048
SCALE_Z = 0.75

frame = EmbryoFrame.from_inference_csv(
    filepath=TARGET_CSV,
    embryo_id="Interactive_Test_Emb",
    time_idx=0, 
    scale_xy=SCALE_XY, 
    scale_z=SCALE_Z
)

## Run single frame alignment

In [23]:
runner = InferenceRunner(engine=engine, oracle=oracle, life_history_df = life_history_df)

# Note: run_for_report takes a list of frames and returns a list of reports
reports = runner.run_for_report([frame])

Generating Trace Data:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Trace Data: 100%|██████████| 1/1 [00:00<00:00, 18.26it/s]


In [20]:
# Alignment Output
# Extract the single frame report
report = reports[0]

# 1. Top-Level Metadata
print(f"Embryo ID:      {report['embryo_id']}")
print(f"Time Index:     {report['time_idx']}")
print(f"Observed Cells: {report['num_cells']}")

# 2. Extract the winning alignment 
best = report['best_result']

print(f"Winning Slice ID: {best['slice_id']}")
print(f"MAP Time Estimate: {best['map_time']:.1f} minutes")
print(f"Total Sinkhorn Cost: {best['cost']:.1f}")
print(f"Optimal Initial Roll Angle: {best['init_angle']}°")

# Extract the final 3D coordinates and lineage assignments
predicted_labels = best['labels']
aligned_xyz = best['coords']

print(f"\nPredicted Labels: {predicted_labels}")
print(f"Aligned Coordinates: {aligned_xyz}")

# 3. Assess alignment confidence
print(f"Mean Frame Confidence: {best.get('mean_confidence', 0.0):.1%}")

# Inspect the cell level diagnostic DataFrame
diag_df = best['diagnostics']
display(diag_df.head())



Embryo ID:      Interactive_Test_Emb
Time Index:     0
Observed Cells: 12
Winning Slice ID: 8
MAP Time Estimate: 41.3 minutes
Total Sinkhorn Cost: 26.4
Optimal Initial Roll Angle: 48.00000000000001°

Predicted Labels: ['ABalp', 'MS', 'E', 'ABplp', 'ABpla', 'P3', 'ABala', 'ABprp', 'ABarp', 'ABara', 'C', 'ABpra']
Aligned Coordinates: [[1.37785694 1.62231471 0.9086726 ]
 [2.24264904 1.73218623 1.17046075]
 [2.86004098 1.44828671 0.87519479]
 [2.10688657 1.22065568 0.59432646]
 [1.66874499 1.09558554 0.59116443]
 [3.37779426 1.16753002 1.02964553]
 [1.13256809 1.39894701 1.05046683]
 [2.78804361 1.25086296 1.50035684]
 [1.91367495 0.83789867 1.23131907]
 [1.63371634 1.04042297 1.5501652 ]
 [2.74766502 0.70774022 1.09751431]
 [2.4852551  1.12503441 1.55869069]]
Mean Frame Confidence: 99.5%


,frame_id,embryo_id,time_idx,cell_name,lineage,x_aligned,y_aligned,z_aligned,mah_dist,entropy,map_time,frame_is_generated,num_cells_in_frame,div_delta,is_correct,confidence_score
0,Interactive_Test_Emb_tid0,Interactive_Test_Emb,0,ABalp,AB,1.377857,1.622315,0.908673,1.259392,6.133077e-09,41.285714,False,12,0.082975,False,0.997314
1,Interactive_Test_Emb_tid0,Interactive_Test_Emb,0,MS,MS,2.242649,1.732186,1.170461,1.549684,3.149413e-10,41.285714,False,12,0.838509,False,0.995334
2,Interactive_Test_Emb_tid0,Interactive_Test_Emb,0,E,E,2.860041,1.448287,0.875195,1.572818,3.149820e-10,41.285714,False,12,0.675901,False,0.993109
3,Interactive_Test_Emb_tid0,Interactive_Test_Emb,0,ABplp,AB,2.106887,1.220656,0.594326,1.239295,4.690388e-10,41.285714,False,12,0.073661,False,0.997314
4,Interactive_Test_Emb_tid0,Interactive_Test_Emb,0,ABpla,AB,1.668745,1.095586,0.591164,1.620065,3.398592e-10,41.285714,False,12,0.079454,False,0.997023


In [ ]:
# Generate HTML alignment report
report_builder = HTMLReportBuilder(growth_df=pd.read_csv('data/empirical_growth_curve.csv'))

report_builder.build_report(reports[0], output_path="data/alignment_report.html")

'data/alignment_report.html'